In [1]:
import os
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader
from cbir.dataset import CbirDataset
from cbir.models import DatabaseFeatureExtractor

In [2]:
BATCH_SIZE = 60
OXFORD_ROOT = "/home/ubuntu/data/datasets/roxford5k/jpg"
CACHE_DIR = "/home/ubuntu/data/feature_cache"

In [ ]:
oxford_dataset = CbirDataset(OXFORD_ROOT)
oxford_dataloader = DataLoader(oxford_dataset, batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
db_extractor = DatabaseFeatureExtractor(num_features=500, num_clusters=10).to(device)
db_extractor.eval()

with torch.no_grad():
    print(f"Processing roxford5k....")
    oxford_feature_cache = {}

    for batch_images, batch_ids in tqdm(oxford_dataloader):
        batch_images = batch_images.to(device)
        
        batched_clusters, batched_centroids, batched_radii = db_extractor(batch_images)
        
        batched_clusters = batched_clusters.cpu()
        batched_centroids = batched_centroids.cpu()
        batched_radii = batched_radii.cpu()

        for i in range(len(batch_ids)):
            image_id = batch_ids[i]

            oxford_feature_cache[image_id] = {
                'clusters': batched_clusters[i],
                'centroid': batched_centroids[i],
                'radius': batched_radii[i]
            }

    torch.save(oxford_feature_cache, os.path.join(CACHE_DIR, "roxford5k_features_pruned.pkl"))

DatabaseFeatureExtractor(
  (resnet_backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
 